# EXA-STAR: neuro-evolved ViT-MAE foundation model on HCP fMRI (Kaggle T4x2)

Evolves a masked-autoencoder vision transformer whose encoder/decoder internals are an EXAMM-style
block graph, trained on the full subject-organized HCP corpus with a subject-level 70/10/20 split.

**Setup expected:**
- The HCP corpus (subject subdirs of `.npz`) mounted as a **Kaggle Dataset** at `HCP_ROOT`.
- The exa-star repo importable (clone it or mount it as a dataset; see the imports cell).

**Kaggle notes:** sessions time out (~12h) and GPU quota is weekly, so the loop **checkpoints to
`/kaggle/working` and resumes** automatically. This first version trains on a **single T4**; two-GPU
genome-parallelism is a later enhancement (see the final markdown cell).


## 1. Configuration

In [ ]:
import os

# --- repo + data locations (EDIT these) ---
REPO_PATH = "/kaggle/working/exa-star"                 # where the repo is cloned on Kaggle
REPO_URL = "https://github.com/axj2613/exa-ae.git"     # your fork; add a token here if private
REPO_BRANCH = "autoencoder-aryan"
HCP_ROOT = "/kaggle/input/hcp-aal424/aal_424"          # dir containing the <subject_id>/ subdirs
ATLAS_COORDS = os.path.join(REPO_PATH, "datasets/hcp/atlases/A424_Coordinates.dat")

WORKDIR = "/kaggle/working"
SPLIT_PATH = os.path.join(WORKDIR, "subject_split.json")     # persisted so resume/eval reuse the SAME split
STATS_PATH = os.path.join(WORKDIR, "norm_stats.npz")         # frozen train-split normalization
LENGTH_INDEX_PATH = os.path.join(WORKDIR, "length_index.json")  # cached recording lengths (header-only scan)
CHECKPOINT_PATH = os.path.join(WORKDIR, "evolution_checkpoint.pkl")
BEST_GENOME_PATH = os.path.join(WORKDIR, "best_genome.pkl")

# --- windowing ---
# A handful of HCP runs are truncated (some tasks are as short as ~35 timepoints), so no single
# window can include literally every recording without being uselessly tiny. window=120 (6 temporal
# patches) includes 99.7% of recordings -- all complete runs incl. EMOTION -- and the dataset logs
# how many truncated recordings (<window) it excludes. Raise toward 140/160 for more temporal
# context at the cost of dropping a few more short runs.
WINDOW_LENGTH = 120          # must be divisible by TIME_PATCH_SIZE
TIME_PATCH_SIZE = 20         # 120 / 20 = 6 temporal patches per parcel
MASK_RATIO = 0.75
SPLIT_RATIOS = (0.7, 0.1, 0.2)

# --- model / evolution ---
D_MODEL = 128
NUM_HEADS = 4
D_FF = 256
DROPOUT = 0.1
# full mixed-cell search, or e.g. ["attention"] to compare an attention-only model
NODE_TYPES = ["attention", "simple", "sequence_lstm", "temporal_lstm"]
POPULATION_SIZE = 10
NUM_GENERATIONS = 300

# --- per-genome training budget ---
NUM_ITERATIONS = 2
BATCHES_PER_ITERATION = 20
BATCH_SIZE = 8
FITNESS_BATCHES = 8
LEARNING_RATE = 0.001
USE_AMP = True               # CUDA mixed precision (tensor cores on T4)

CHECKPOINT_EVERY = 5         # genomes between checkpoints


## 2. Fetch the repo + imports

Clones the repo on a fresh session (or pulls the latest on re-run) so the code always matches
this notebook -- no more manual dataset re-uploads. If your fork is **private**, put a GitHub
token in `REPO_URL` (e.g. from Kaggle Secrets): `https://{token}@github.com/axj2613/exa-ae.git`.

*(If you `git pull` new code into an already-imported session, restart the kernel so Python
reloads the modules.)*

In [ ]:
import os
import subprocess
import sys

if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull", "origin", REPO_BRANCH], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import pickle
import torch
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


## 3. Dataset

Builds the subject-level split (persisted) and freezes per-parcel normalization from the training
split (persisted). Both files are reused on resume and by the downstream embedding extraction so
the split and normalization are identical everywhere.

In [ ]:
from time_series.hcp_window_dataset import HCPWindowDataset

dataset = HCPWindowDataset(
    root_dir=HCP_ROOT,
    atlas_coordinates_filename=ATLAS_COORDS,
    window_length=WINDOW_LENGTH,
    split_ratios=SPLIT_RATIOS,
    split_path=SPLIT_PATH,
    stats_path=STATS_PATH,
    length_index_path=LENGTH_INDEX_PATH,
)
print("parcels:", dataset.num_parcels)
print("split sizes:", {k: len(v) for k, v in dataset.splits.items()})


## 4. Build or resume the population

If a checkpoint exists (from a previous timed-out session) it is loaded and the run continues;
otherwise a fresh seed genome + population is created.

In [ ]:
from population.single_population import SinglePopulation
from evolution.vision_transformer_block_edge_generator import VisionTransformerBlockEdgeGenerator
from evolution.vision_transformer_block_node_generator import VisionTransformerBlockNodeGenerator
from evolution.vision_transformer_block_reproduction_selector import VisionTransformerBlockReproductionSelector
from genomes.vision_transformer_block_genome import VisionTransformerBlockGenome
from weight_generators.lamarckian_block_weight_generator import LamarckianBlockWeightGenerator
from evolution.checkpoint import save_checkpoint, load_checkpoint

if os.path.exists(CHECKPOINT_PATH):
    state = load_checkpoint(CHECKPOINT_PATH)
    population = state["population_strategy"]
    start_generation = state["generation"]
    # the checkpoint was pickled on CPU; re-home the whole population (and seed) to the GPU so
    # it isn't device-mixed with the GPU children generated after resuming.
    for genome in population.population:
        genome.to(device)
    if population.seed_genome is not None:
        population.seed_genome.to(device)
    print(f"resumed from checkpoint at generation {start_generation}")
else:
    weight_generator = LamarckianBlockWeightGenerator()
    node_generator = VisionTransformerBlockNodeGenerator(
        num_heads=NUM_HEADS, d_ff=D_FF, dropout=DROPOUT, allowed_node_types=NODE_TYPES
    )
    edge_generator = VisionTransformerBlockEdgeGenerator()

    seed_genome = VisionTransformerBlockGenome(
        generation_number=0, num_parcels=dataset.num_parcels, window_length=WINDOW_LENGTH,
        parcel_coordinates=dataset.parcel_coordinates, d_model=D_MODEL, num_heads=NUM_HEADS,
        d_ff=D_FF, dropout=DROPOUT, time_patch_size=TIME_PATCH_SIZE, mask_ratio=MASK_RATIO,
        weight_generator=weight_generator,
    )
    population = SinglePopulation(
        population_size=POPULATION_SIZE, seed_genome=seed_genome,
        reproduction_selector=VisionTransformerBlockReproductionSelector(
            node_generator=node_generator, edge_generator=edge_generator, weight_generator=weight_generator,
        ),
    )
    start_generation = 0
    print("starting fresh evolution with node types:", NODE_TYPES)


## 5. Evolution loop

Each generation: generate a child genome, move it to the GPU, train it on the **train** split
(mixed precision), score its fitness on the **validation** split, and insert it. Checkpoints every
`CHECKPOINT_EVERY` genomes survive session restarts.

*(The reproduction operators print verbosely; filter stdout/loguru if you want a quieter log.)*

In [ ]:
config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, tuple, list))}

for generation in tqdm(range(start_generation, NUM_GENERATIONS), initial=start_generation, total=NUM_GENERATIONS):
    genome = population.generate_genome()
    genome.to(device)                                   # re-home CPU-generated nodes/edges to GPU
    optimizer = torch.optim.Adam(genome.parameters(), lr=LEARNING_RATE)
    genome.train(
        dataset=dataset, optimizer=optimizer, iterations=NUM_ITERATIONS,
        batch_size=BATCH_SIZE, batches_per_iteration=BATCHES_PER_ITERATION,
        fitness_batches=FITNESS_BATCHES, use_amp=USE_AMP,
    )
    population.insert_genome(genome)

    if (generation + 1) % CHECKPOINT_EVERY == 0:
        # save_checkpoint moves all genomes to CPU (portable); the next generate_genome deepcopies
        # CPU parents and genome.to(device) re-homes the child, so the loop self-heals.
        save_checkpoint(CHECKPOINT_PATH, population, generation + 1, config=config)
        best = population.population[0]
        with open(BEST_GENOME_PATH, "wb") as best_file:
            pickle.dump(best, best_file)
        print(f"[checkpoint @ gen {generation + 1}] best validation MSE: {best.fitness:.6f}")


## 6. Final save + download

All outputs are written to `/kaggle/working` (`WORKDIR`). **`/kaggle/working` is wiped when an
interactive session ends unless you persist it** -- so to keep the trained model do ONE of:
- enable **Persistence -> Files only** in the notebook settings (right sidebar) -- also required
  for the checkpoint/resume across sessions to work; or
- **Save Version** (Save & Run All), which stores `/kaggle/working` as the version's Output; or
- download the files now via the links printed below.

Downstream clinical evaluation (`subject_embeddings.py` -> `eval_cls.py`) needs THREE of these,
not just the genome: `best_genome.pkl`, `subject_split.json`, and `norm_stats.npz` -- so that the
held-out test subjects and the normalization match what the model was trained with.

In [ ]:
from IPython.display import FileLink, display

save_checkpoint(CHECKPOINT_PATH, population, NUM_GENERATIONS, config=config)
best = population.population[0]
with open(BEST_GENOME_PATH, "wb") as best_file:
    pickle.dump(best, best_file)

print("best genome validation MSE:", best.fitness)
print(best)

print("\nartifacts in", WORKDIR, "(download these -- the first three are needed for clinical eval):")
for path in [BEST_GENOME_PATH, SPLIT_PATH, STATS_PATH, CHECKPOINT_PATH, LENGTH_INDEX_PATH]:
    if os.path.exists(path):
        print(f"  {os.path.getsize(path) / 1e6:8.2f} MB  {path}")
        display(FileLink(os.path.relpath(path, WORKDIR)))


## Later: two-GPU (T4x2) genome-parallelism

Genome training is independent, so throughput ~doubles by training two genomes at once, one per
T4. The clean way with this sequential population is **generational batching**: generate K genomes,
train them concurrently across `cuda:0`/`cuda:1` (one worker per device), then insert all K. This
slots around the loop above without changing the genome or dataset code; add it once the
single-GPU run is validated.